# TP — Segmentation d'Images Satellites avec U-Net

**Module** : Systèmes Répartis et Programmation Parallèle  
**Master S2 — Intelligence Artificielle** · Faculté des Sciences Ben M'sick  
**Groupe présentateur** : GT019 — Anass Dabibe · Hassan El Hadi · Marouane Ait Tamanait

---

## Objectif

Entraîner un réseau **U-Net** pour la segmentation sémantique d'images satellites (aériennes de Dubaï) sur **GPU**, et comprendre comment le **parallélisme de données** (GPU, `DataParallel`, DDP) accélère l'entraînement.

## Comment exécuter ce TP (sur Kaggle)

1. Créez un compte gratuit sur [kaggle.com](https://www.kaggle.com) et vérifiez votre email (nécessaire pour le GPU).
2. Ouvrez le dataset : [Semantic Segmentation of Aerial Imagery](https://www.kaggle.com/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery) puis **New Notebook** (le dataset est attaché automatiquement). Sinon, dans le panneau droit → **+ Add Input** → cherchez `semantic-segmentation-of-aerial-imagery`.
3. Importez ce notebook : **File → Import Notebook**.
4. Activez le GPU : **Session Options → Accelerator → GPU T4 x2 → Save**.
5. Cliquez **Run All** et exécutez les cellules dans l'ordre.

> ⚠️ **Attention** : si la première cellule affiche `Device : cpu`, le GPU n'est pas activé — retournez à l'étape 4.

**Durée estimée** : ~1h30 (dont 15–20 min d'entraînement sur GPU T4).

Au fil du TP, vous devrez réaliser **6 captures d'écran** (signalées par 📸) et répondre aux **questions de réflexion** en fin de notebook. Le tout est à rendre dans le PDF `QUESTIONS_TP.pdf`.

## 0 — Chemins et détection du dataset

In [ ]:
# ── Paths ────────────────────────────────────
import os

KAGGLE_INPUT = '/kaggle/input/semantic-segmentation-of-aerial-imagery'
SAVE_DIR     = '/kaggle/working'

# Affiche le contenu du dossier d'entree pour reperer les noms exacts
print('Input folder contents:')
for root, dirs, files in os.walk(KAGGLE_INPUT):
    depth = root.replace(KAGGLE_INPUT, '').count(os.sep)
    print('  ' * depth + os.path.basename(root) + '/')
    if depth >= 2:          # stop apres 2 niveaux
        dirs.clear()

# Auto-detection : trouve le dossier qui contient directement les sous-dossiers 'Tile X'
DATA_ROOT = None
for root, dirs, files in os.walk(KAGGLE_INPUT):
    if any(d.lower().startswith('tile') for d in dirs):
        DATA_ROOT = root
        break

assert DATA_ROOT is not None, (
    'Could not find Tile folders inside ' + KAGGLE_INPUT +
    '\nMake sure the dataset is attached via + Add Input'
)

CHECKPOINT = SAVE_DIR + '/best_unet.pth'
FIG_PATH   = SAVE_DIR + '/predictions.png'

print('\nData root :', DATA_ROOT)
print('Contents  :', sorted(os.listdir(DATA_ROOT))[:6])

In [ ]:
# ── Imports ─────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch :', torch.__version__)
print('Device  :', device)
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

## 1 — Dataset

In [ ]:
CLASS_NAMES  = ['Building', 'Land', 'Road', 'Vegetation', 'Water', 'Unlabeled']
CLASS_COLORS = [
    (226, 169,  41),
    (132,  41, 246),
    (110, 193, 228),
    ( 60,  16, 152),
    (254, 221,  58),
    (155, 155, 155),
]
COLORS_NORM = [(r/255, g/255, b/255) for r, g, b in CLASS_COLORS]


def rgb_to_mask(mask_pil, img_size):
    mask = np.array(mask_pil.resize((img_size, img_size), Image.NEAREST))
    out  = np.full(mask.shape[:2], 5, dtype=np.int64)
    for idx, color in enumerate(CLASS_COLORS):
        out[np.all(mask == color, axis=-1)] = idx
    return torch.tensor(out, dtype=torch.long)


class DubaiDataset(Dataset):
    def __init__(self, root, img_size=256):
        self.img_size = img_size
        self.samples  = []
        self.tf = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        for tile in sorted(os.listdir(root)):
            img_dir = os.path.join(root, tile, 'images')
            msk_dir = os.path.join(root, tile, 'masks')
            if not os.path.isdir(img_dir):
                continue
            for f in sorted(os.listdir(img_dir)):
                mp = os.path.join(msk_dir, os.path.splitext(f)[0] + '.png')
                if os.path.exists(mp):
                    self.samples.append((os.path.join(img_dir, f), mp))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        img_path, msk_path = self.samples[i]
        img  = self.tf(Image.open(img_path).convert('RGB'))
        mask = rgb_to_mask(Image.open(msk_path).convert('RGB'), self.img_size)
        return img, mask


dataset = DubaiDataset(DATA_ROOT)
print(f'Samples : {len(dataset)}')
img, mask = dataset[0]
print(f'Image   : {img.shape}  Mask: {mask.shape}  Classes: {mask.unique().tolist()}')

> 📸 **Capture d'écran #1** — Exécutez la cellule ci-dessous et capturez la grille **4 images / 4 masques**.

In [ ]:
# Visualise quelques paires image / masque
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img, mask = dataset[i * (len(dataset) // 4)]
    img_disp  = (img * STD + MEAN).permute(1, 2, 0).numpy().clip(0, 1)
    msk_rgb   = np.zeros((*mask.shape, 3))
    for c, col in enumerate(COLORS_NORM):
        msk_rgb[mask.numpy() == c] = col
    axes[0, i].imshow(img_disp); axes[0, i].set_title(f'Image {i}');  axes[0, i].axis('off')
    axes[1, i].imshow(msk_rgb);  axes[1, i].set_title(f'Masque {i}'); axes[1, i].axis('off')

patches = [mpatches.Patch(color=c, label=n) for c, n in zip(COLORS_NORM, CLASS_NAMES)]
fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=10)
plt.suptitle('Exemples du dataset Dubai (Humans in the Loop, CC0)', fontsize=13)
plt.tight_layout()
plt.savefig(SAVE_DIR + '/samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 2 — Architecture U-Net

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=6, features=[64, 128, 256, 512]):
        super().__init__()
        self.encoders   = nn.ModuleList()
        self.decoders   = nn.ModuleList()
        self.pool       = nn.MaxPool2d(2, 2)
        ch = in_channels
        for f in features:
            self.encoders.append(DoubleConv(ch, f))
            ch = f
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        for f in reversed(features):
            self.decoders.append(nn.ConvTranspose2d(f * 2, f, 2, 2))
            self.decoders.append(DoubleConv(f * 2, f))
        self.head = nn.Conv2d(features[0], num_classes, 1)

    def forward(self, x):
        skips = []
        for enc in self.encoders:
            x = enc(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        for i in range(0, len(self.decoders), 2):
            x = self.decoders[i](x)
            s = skips[-(i // 2 + 1)]
            if x.shape != s.shape:
                x = nn.functional.interpolate(x, size=s.shape[2:])
            x = self.decoders[i + 1](torch.cat([s, x], dim=1))
        return self.head(x)


model = UNet().to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Parametres : {total:,}')

with torch.no_grad():
    out = model(torch.randn(2, 3, 256, 256, device=device))
print(f'Output     : {out.shape}  (attendu [2, 6, 256, 256])')

## 3 — Entraînement

> 📸 **Capture d'écran #2** — Pendant l'entraînement, capturez les **premières et dernières époques** montrant la progression de la Loss et de l'IoU.

In [ ]:
# ── Hyperparametres ───────────────────────────
EPOCHS      = 30
BATCH_SIZE  = 8
LR          = 1e-3
NUM_CLASSES = 6

# ── Split train / val ────────────────────────
n_val   = max(1, int(0.15 * len(dataset)))
n_train = len(dataset) - n_val
train_set, val_set = random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train : {n_train}  Val : {n_val}  Batches/epoch : {len(train_loader)}')

# ── Modele + DataParallel si 2 GPU ─────────────
model = UNet(num_classes=NUM_CLASSES).to(device)
if torch.cuda.device_count() > 1:
    print(f'DataParallel sur {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(model)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

In [ ]:
def mean_iou(preds, masks):
    pred_cls = preds.argmax(dim=1)
    ious = []
    for c in range(NUM_CLASSES - 1):   # ignore Unlabeled
        inter = ((pred_cls == c) & (masks == c)).sum().float()
        union = ((pred_cls == c) | (masks == c)).sum().float()
        if union > 0:
            ious.append((inter / union).item())
    return float(np.mean(ious)) if ious else 0.0


train_losses, val_losses, val_ious = [], [], []
best_iou = 0.0

for epoch in range(1, EPOCHS + 1):

    # ── Train ──
    model.train()
    t_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'Epoch {epoch:02d}/{EPOCHS}', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
    t_loss /= len(train_loader)

    # ── Validate ──
    model.eval()
    v_loss, v_iou = 0.0, 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds  = model(imgs)
            v_loss += criterion(preds, masks).item()
            v_iou  += mean_iou(preds, masks)
    v_loss /= len(val_loader)
    v_iou  /= len(val_loader)
    scheduler.step(v_loss)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    val_ious.append(v_iou)

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss : {t_loss:.4f} | '
          f'Val Loss : {v_loss:.4f} | '
          f'Val IoU : {v_iou:.4f}')

    if v_iou > best_iou:
        best_iou = v_iou
        torch.save(model.state_dict(), CHECKPOINT)
        print(f'  => checkpoint sauvegarde  (IoU={best_iou:.4f})')

print(f'\nMeilleur IoU : {best_iou:.4f}')

## 4 — Courbes d'apprentissage

> 📸 **Capture d'écran #3** — Capturez les **deux courbes d'apprentissage** (Loss et IoU).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(train_losses, 'o-', label='Train Loss')
ax1.plot(val_losses,   's-', label='Val Loss')
ax1.set_xlabel('Epoque'); ax1.set_ylabel('CrossEntropy Loss')
ax1.set_title('Courbes de Loss'); ax1.legend(); ax1.grid(True)

ax2.plot(val_ious, '^-', color='green', label='Val Mean IoU')
ax2.set_xlabel('Epoque'); ax2.set_ylabel('Mean IoU')
ax2.set_title('IoU de Validation'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig(SAVE_DIR + '/training_curves.png', dpi=150)
plt.show()

## 5 — Évaluation (IoU par classe)

> 📸 **Capture d'écran #4** — Capturez le **tableau d'IoU par classe**.

In [ ]:
# Charger le meilleur checkpoint
state = torch.load(CHECKPOINT, map_location=device)
state = {k.replace('module.', ''): v for k, v in state.items()}  # enlever prefixe DataParallel
best_model = UNet(num_classes=NUM_CLASSES).to(device)
best_model.load_state_dict(state)
best_model.eval()

inter = [0.0] * NUM_CLASSES
union = [0.0] * NUM_CLASSES

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = best_model(imgs).argmax(dim=1)
        for c in range(NUM_CLASSES):
            inter[c] += ((preds == c) & (masks == c)).sum().item()
            union[c] += ((preds == c) | (masks == c)).sum().item()

print(f'{"Classe":<14} {"IoU":>6}')
print('-' * 22)
ious = []
for c in range(NUM_CLASSES - 1):
    iou = inter[c] / union[c] if union[c] > 0 else float('nan')
    ious.append(iou)
    print(f'{CLASS_NAMES[c]:<14} {iou:.4f}')
print('-' * 22)
print(f'{"Mean IoU":<14} {np.nanmean(ious):.4f}')

## 6 — Visualisation des prédictions

> 📸 **Capture d'écran #5** — Capturez la **figure de prédictions** complète (4 lignes × 3 colonnes : Image | Vérité terrain | Prédiction U-Net).

In [ ]:
imgs_b, masks_b = next(iter(val_loader))
with torch.no_grad():
    preds_b = best_model(imgs_b.to(device)).cpu()
pred_cls = preds_b.argmax(dim=1)

n = min(4, len(imgs_b))
fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
for i in range(n):
    img_disp = (imgs_b[i] * STD + MEAN).permute(1, 2, 0).numpy().clip(0, 1)
    gt_rgb   = np.zeros((256, 256, 3))
    pr_rgb   = np.zeros((256, 256, 3))
    for c, col in enumerate(COLORS_NORM):
        gt_rgb[masks_b[i].numpy() == c]   = col
        pr_rgb[pred_cls[i].numpy() == c]  = col
    axes[i, 0].imshow(img_disp); axes[i, 0].set_title('Image satellite'); axes[i, 0].axis('off')
    axes[i, 1].imshow(gt_rgb);   axes[i, 1].set_title('Verite terrain');  axes[i, 1].axis('off')
    axes[i, 2].imshow(pr_rgb);   axes[i, 2].set_title('Prediction U-Net');axes[i, 2].axis('off')

patches = [mpatches.Patch(color=c, label=n) for c, n in zip(COLORS_NORM, CLASS_NAMES)]
fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=9, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout()
plt.savefig(FIG_PATH, bbox_inches='tight', dpi=150)
plt.show()
print('Figure sauvegardee :', FIG_PATH)

## 7 — Benchmark GPU vs CPU

> 📸 **Capture d'écran #6** — Capturez le **résultat du benchmark** CPU/GPU et son facteur d'accélération.

In [ ]:
import time

batch   = torch.randn(8, 3, 256, 256)
m_cpu   = UNet().eval()

t0 = time.time()
with torch.no_grad():
    _ = m_cpu(batch)
cpu_ms = (time.time() - t0) * 1000
print(f'CPU  -- forward pass (batch=8) : {cpu_ms:.1f} ms')

if torch.cuda.is_available():
    m_gpu  = UNet().to('cuda').eval()
    b_gpu  = batch.cuda()
    with torch.no_grad(): _ = m_gpu(b_gpu)   # warmup
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad(): _ = m_gpu(b_gpu)
    torch.cuda.synchronize()
    gpu_ms = (time.time() - t0) * 1000
    print(f'GPU  -- forward pass (batch=8) : {gpu_ms:.1f} ms')
    print(f'Acceleration GPU : {cpu_ms / gpu_ms:.1f}x')

## 8 — Questions de réflexion

Répondez à ces questions dans votre rapport PDF (voir aussi `QUESTIONS_TP.pdf`).

**Q1 — Architecture.** Pourquoi U-Net utilise-t-il des *skip connections* ? Que se passerait-il si on les supprimait ?

**Q2 — Métrique.** Pourquoi utilise-t-on l'IoU (*Intersection over Union*) plutôt que l'accuracy pour évaluer la segmentation ?

**Q3 — Parallélisme.** Quelle est la différence entre `DataParallel` et `DistributedDataParallel` (DDP) dans PyTorch ? Lequel est plus efficace et pourquoi ?

**Q4 — Résultats.** Quelle classe obtient le meilleur IoU dans vos résultats ? Proposez une explication basée sur les caractéristiques visuelles de cette classe.

**Q5 — GPU.** D'après votre benchmark, quel est le facteur d'accélération GPU obtenu ? En quoi cela justifie-t-il l'usage des systèmes répartis pour l'entraînement de modèles de deep learning ?

---

### Livrable attendu

Un fichier **PDF unique** nommé `TP_GT0XX_NomPrenom.pdf` contenant :
1. Les **6 captures d'écran** (📸 #1 à #6), chacune avec une brève description (2–3 phrases).
2. Les **réponses** aux 5 questions de réflexion (Q1 à Q5).
3. La figure **prédictions** en grand format.

| Élément | Points |
|---|---|
| Captures d'écran (6 × 1 pt) | 6 |
| Questions Q1–Q5 (5 × 2 pts) | 10 |
| Qualité de la rédaction | 4 |
| **Total** | **20** |

_Bon courage ! — Groupe GT019 — Systèmes Répartis & GPU 2025–2026_